<a href="https://colab.research.google.com/github/tejamuni326/250200340_Teja_Sec1_Week13/blob/main/2502003340_Teja_Sec1_Week13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 pandas supabase psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 51.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [supabase]


In [2]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from IPython.display import display

In [3]:
url = "https://www.bikehirechennai.com/bikes.php"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=30)

print("Status Code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

Status Code: 200


In [4]:
bikes = []

for card in soup.find_all(["div", "article"]):
    text = card.get_text(" ", strip=True)

    title = card.find(["h2", "h3"])
    rent = re.search(r"₹\s*([\d,]+)\s*/\s*day", text, re.I)
    engine = re.search(r"Engine\s*(\d+)\s*cc", text, re.I)
    gearbox = re.search(r"Gearbox\s*([A-Za-z]+)", text, re.I)

    if title and rent and engine and gearbox:
        bikes.append({
            "Title": title.get_text(strip=True),
            "Rent_Per_Day": int(rent.group(1).replace(",", "")),
            "Engine_CC": int(engine.group(1)),
            "Gearbox": gearbox.group(1)
        })

df = pd.DataFrame(bikes)

print("Total Bikes:", len(df))
display(df.head(10))

Total Bikes: 126


,Title,Rent_Per_Day,Engine_CC,Gearbox
0,Filters,15000,110,Automatic
1,Filters,15000,110,Automatic
2,Honda Activa 2023–2025,330,110,Automatic
3,Honda Activa 2023–2025,330,110,Automatic
4,Honda Activa 2023–2025,330,110,Automatic
5,Honda Activa 2023–2025,330,110,Automatic
6,Honda Activa 6G BS6 (2020-2023),300,110,Automatic
7,Honda Activa 6G BS6 (2020-2023),300,110,Automatic
8,Honda Activa (2015–2019),250,110,Automatic
9,Honda Activa (2015–2019),250,110,Automatic


In [5]:
df.to_csv("bike_rental_data.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


In [12]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from getpass import getpass

DB_PASSWORD = getpass("Enter Supabase database password: ")

DB_USER = "postgres.afjqobsrazkjjalplfjd"
DB_HOST = "aws-1-ap-northeast-1.pooler.supabase.com"
DB_PORT = "5432"
DB_NAME = "postgres"

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:"
    f"{quote_plus(DB_PASSWORD)}@{DB_HOST}:"
    f"{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

with engine.connect() as connection:
    print("Connected to Supabase successfully!")

Enter Supabase database password: ··········
Connected to Supabase successfully!


In [13]:
df_upload = df.rename(columns={
    "Title": "title",
    "Rent_Per_Day": "rent_per_day",
    "Engine_CC": "engine_cc",
    "Gearbox": "gearbox"
})

df_upload.to_sql(
    name="bike_rentals",
    con=engine,
    schema="public",
    if_exists="append",
    index=False
)

print("Bike data uploaded successfully!")

Bike data uploaded successfully!


In [14]:
query = "SELECT * FROM bike_rentals;"

df_supabase = pd.read_sql(query, engine)

display(df_supabase)

,id,title,rent_per_day,engine_cc,gearbox
0,1,Filters,15000,110,Automatic
1,2,Filters,15000,110,Automatic
2,3,Honda Activa 2023–2025,330,110,Automatic
3,4,Honda Activa 2023–2025,330,110,Automatic
4,5,Honda Activa 2023–2025,330,110,Automatic
...,...,...,...,...,...
121,122,Royal Enfield RE Classic 350cc,620,350,Manual
122,123,Royal Enfield RE Thunderbird 350X,620,350,Manual
123,124,Royal Enfield RE Thunderbird 350X,620,350,Manual
124,125,Harley Davidson Sportster 48,3200,1200,Manual


Distribution of bikes by gearbox type

In [15]:
query = """
SELECT gearbox, COUNT(*) AS total_bikes
FROM bike_rentals
GROUP BY gearbox;
"""

pd.read_sql(query, engine)

,gearbox,total_bikes
0,Manual,84
1,Automatic,42


Which gearbox type has the highest average rent per day?

In [16]:
query = """
SELECT gearbox, ROUND(AVG(rent_per_day), 2) AS average_rent
FROM bike_rentals
GROUP BY gearbox
ORDER BY average_rent DESC
LIMIT 1;
"""

pd.read_sql(query, engine)

,gearbox,average_rent
0,Automatic,987.62


Correlation between engine capacity and rent per day

In [17]:
query = """
SELECT CORR(engine_cc, rent_per_day) AS correlation
FROM bike_rentals;
"""

pd.read_sql(query, engine)

,correlation
0,0.123347


Count bikes with engine capacity greater than 150 cc

In [18]:
query = """
SELECT COUNT(*) AS total_bikes
FROM bike_rentals
WHERE engine_cc > 150;
"""

pd.read_sql(query, engine)

,total_bikes
0,56
